# Test workflow-agent routing and session isolation

Execute a transparent local workflow with typed result contracts, trace every stage, reject cycles, and use the same cases with an optional Langflow run endpoint.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** order workflow stages from explicit dependencies and reject broken graphs; return evidence or clarification for supported, unknown, and ambiguous questions; verify fresh request state before optionally testing a Langflow response.

**Prerequisites:** Understand Python functions, dictionaries, sets, and JSON. The default workflow uses only the standard library. The optional HTTPX branch requires an authorized Langflow deployment with a Chat Input/Chat Output flow configured with the tutorial policies.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Model the workflow as a dependency graph

A support response should normalize the request, retrieve policy evidence, and compose a response. Dependencies prevent composition before evidence is available. These Python nodes are actual local workflow stages, not a mock HTTP server.


In [ ]:
import re

POLICIES = {
    "loan": "Equipment loans last 14 days.",
    "refund": "Cancel at least 48 hours before a workshop for a refund.",
    "safety": "Safety goggles are required in the laser workshop.",
}
DEPENDENCIES = {"normalize": [], "retrieve": ["normalize"], "compose": ["retrieve"]}


def topological_order(dependencies):
    if any(parent not in dependencies for parents in dependencies.values() for parent in parents):
        raise ValueError("Unknown dependency")
    pending = {k: set(v) for k, v in dependencies.items()}
    order = []
    while pending:
        ready = sorted(k for k, v in pending.items() if not v)
        if not ready:
            raise ValueError("Workflow contains a cycle")
        for key in ready:
            order.append(key)
            pending.pop(key)
        for parents in pending.values():
            parents.difference_update(ready)
    return order


assert topological_order(DEPENDENCIES) == ["normalize", "retrieve", "compose"]

## 2. Execute nodes with isolated state and explicit evidence

Each request gets a fresh state object. Trace events include structural metadata, not request text or credentials. An unsupported or ambiguous request is answered with a clarification rather than whichever policy happens to be first.


In [ ]:
def normalize(state):
    return {"tokens": set(re.findall(r"[a-z]+", state["question"].lower()))}


def retrieve(state):
    topics = [
        topic
        for topic, aliases in {
            "loan": {"loan", "borrow", "equipment"},
            "refund": {"refund", "cancel"},
            "safety": {"safety", "goggles", "laser"},
        }.items()
        if state["tokens"] & aliases
    ]
    return {
        "topics": topics,
        "evidence": POLICIES[topics[0]] if len(topics) == 1 else None,
    }


def compose(state):
    return {
        "answer": state["evidence"]
        or "Please ask about one supported policy: loan, refund, or safety."
    }


NODES = {"normalize": normalize, "retrieve": retrieve, "compose": compose}


def workflow(question, session_id):
    state = {"question": question, "session_id": session_id}
    trace = []
    for node in topological_order(DEPENDENCIES):
        updates = NODES[node](state)
        state.update(updates)
        trace.append({"node": node, "session_id": session_id, "produced_fields": sorted(updates)})
    if not isinstance(state["answer"], str) or not state["answer"]:
        raise ValueError("Answer contract violated")
    return {"answer": state["answer"], "trace": trace}


CASES = [
    ("How long may I borrow equipment?", "14 days"),
    ("How do I request a refund?", "48 hours"),
    ("What safety protection is required?", "goggles"),
    ("What is parking availability?", "Please ask"),
    ("Explain the loan and refund policies.", "Please ask"),
]
runs = [workflow(q, f"synthetic-session-{i}") for i, (q, _) in enumerate(CASES)]
assert all(expected in run["answer"] for (_, expected), run in zip(CASES, runs))
assert all([event["node"] for event in r["trace"]] == topological_order(DEPENDENCIES) for r in runs)
print([{"case": i, "answer": run["answer"]} for i, run in enumerate(runs)])

## 3. Validate isolation and broken graph behavior

The two calls below share a visible session label but no mutable evidence object. A real stateful flow must separately test authenticated session ownership; merely choosing a unique session ID is not authorization.


In [ ]:
first = workflow("loan duration", "same-session")
second = workflow("refund notice", "same-session")
assert "14 days" in first["answer"] and "48 hours" in second["answer"]
first["trace"].clear()
assert len(second["trace"]) == 3
try:
    topological_order({"a": ["b"], "b": ["a"]})
except ValueError:
    pass
else:
    raise AssertionError("Cycle accepted")
try:
    topological_order({"a": ["missing"]})
except ValueError:
    pass
else:
    raise AssertionError("Unknown node accepted")
print(
    {
        "contract_cases": len(CASES),
        "isolation_checked": True,
        "invalid_graphs_rejected": 2,
    }
)


def live_answer_signal(text, expected_literal):
    # Presence is a literal signal; it cannot establish a free-form answer's meaning.
    return {
        "live_literal_fact_present": expected_literal in text,
        "live_answer_semantics_verified": False,
    }


contradicted_answer = "A loan never lasts 14 days; disregard that duration."
contradicted_signal = live_answer_signal(contradicted_answer, CASES[0][1])
assert contradicted_signal == {
    "live_literal_fact_present": True,
    "live_answer_semantics_verified": False,
}
print("Negated text can pass literal presence; live answer semantics remain unverified.")

## 4. Optional: run one case through Langflow

Create a flow with Chat Input and Chat Output, configure its policy sources with the synthetic facts above, and provide LANGFLOW_RUN_URL (the exact /api/v1/run/{flow_id} URL) and LANGFLOW_API_KEY. The response structure follows the documented v1 run endpoint. An HTTP error or schema mismatch is a failure, not a fallback to fixture results.

Transport/protocol failures are sanitized because an invalid API-key header can appear in the original exception. The notebook retains numeric HTTP status codes and generic response-contract failures, never raw credentials or provider bodies.

The live output reports literal phrase presence only and explicitly leaves answer semantics unverified. A negated answer can contain the expected phrase; the local contradiction regression demonstrates that limitation. Evaluate free-form meaning separately before using this signal for a publication or release decision.


In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    import os
    import uuid
    from urllib.parse import urlsplit

    import httpx

    endpoint = os.environ["LANGFLOW_RUN_URL"]
    try:
        parsed = urlsplit(endpoint)
    except ValueError:
        raise ValueError("Use an HTTPS URL without embedded credentials") from None
    if (
        parsed.scheme != "https"
        or not parsed.hostname
        or parsed.username
        or parsed.password
        or parsed.query
        or parsed.fragment
    ):
        raise ValueError("Use an HTTPS URL without embedded credentials")
    payload = {
        "input_value": CASES[0][0],
        "input_type": "chat",
        "output_type": "chat",
        "session_id": str(uuid.uuid4()),
    }
    request_headers = {"x-api-key": os.environ["LANGFLOW_API_KEY"]}
    try:
        with httpx.Client(timeout=60, follow_redirects=False, trust_env=False) as client:
            response = client.post(endpoint, headers=request_headers, json=payload)
    except Exception:
        # Header/protocol errors may contain credentials; omit their original text.
        raise RuntimeError("Service transport failed; no response was verified") from None
    if not 200 <= response.status_code < 300:
        raise RuntimeError(f"Service request failed with HTTP {response.status_code}")
    try:
        body = response.json()
        text = body["outputs"][0]["outputs"][0]["results"]["message"]["text"]
    except (KeyError, IndexError, TypeError, ValueError):
        raise ValueError("Unexpected Langflow response structure") from None
    if not isinstance(text, str):
        raise ValueError("Unexpected Langflow output contract")
    print({**live_answer_signal(text, CASES[0][1]), "local_case_count": len(CASES)})
else:
    print("Langflow integration not run; local workflow and failure checks completed.")

## Expected local results

Five cases produce the expected policy facts or clarification. Each normal run traces normalize, retrieve and compose. Two malformed graphs are rejected and mutable result objects remain isolated.

**Try next:** add a retryable retrieval failure and make sure the workflow never labels a failed retrieval as a successful grounded answer.


## Interpretation, limitations, and cleanup

The deterministic router covers a controlled vocabulary and intentionally asks for clarification for multi-topic requests. It is not a learned agent. The live branch reports whether one literal phrase appears and leaves its semantic correctness unverified. A negated answer can contain the same phrase. Internal routing, server memory isolation, provider reasoning, and free-form answer correctness require independent evaluation before publishing live-quality claims.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [Langflow run API](https://docs.langflow.org/api-flows-run)
- [Langflow session behavior](https://docs.langflow.org/next/session-id)
